# Import Libraries

In [6]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

import pandas as pd
import numpy as np
import re

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

from wordcloud import WordCloud
import matplotlib.pyplot as plt
import seaborn as sns

# Load Dataset

In [2]:
df = pd.read_csv('data_1C.csv', encoding='utf-8')
df.sample(5)

,Unnamed: 0,text,label
5704,5704,Mommy Cuddle Over-The-Belly Maternity Leggings...,Clothing & Accessories
10259,10259,"The Brain: The Story of You Review Nature""An i...",Books
7330,7330,Milton Executive Lunch Box Soft Insulated Tiff...,Household
10683,10683,Classic Accessories 59992 Terrazzo Patio Offse...,Household
12422,12422,American Micronic AMI-TSS2-150Dx 4-Slice 2in1 ...,Household


Remove Unnecessary Columns (Unamed: 0)

In [3]:
df.drop(['Unnamed: 0'], axis=1, inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12606 entries, 0 to 12605
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    12606 non-null  object
 1   label   12606 non-null  object
dtypes: object(2)
memory usage: 197.1+ KB


# Data Preprocessing

## Check for Missing Values

In [4]:
df.isna().sum()

text     0
label    0
dtype: int64

## Lowercase

Lowercase all text to avoid case sensitivity. (e.g. "Hello" and "hello" are considered as different words)

In [21]:
df_preprocessed = df.copy()

df_preprocessed['text'] = df_preprocessed['text'].apply(lambda x: x.lower())

## Remove Punctuation

Remove all punctuation from the text.

In [22]:
#remove punctuation wirh string.punctuation
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

df_preprocessed['text'] = df_preprocessed['text'].apply(lambda x: x.translate(str.maketrans('', '', string.punctuation)))
df_preprocessed.sample(5)

,text,label
10751,what are you doing with your life review one o...,Books
7044,treo borosilicate vector liqueur glass set 255...,Household
7747,jdx micro fibre silknise cushion filler 40x40c...,Household
2702,collins classics – the complete works of willi...,Books
5236,mintkraft foam sheet multicolor self adhesive ...,Books


## Remove Non Alphabetic Characters

Remove all non-alphabetic characters from the text. (e.g. "Hello123" will be converted to "Hello")

In [23]:
#remove non-alphabetic characters
df_preprocessed['text'] = [re.sub('[^a-zA-Z]', ' ', text) for text in df_preprocessed['text']]

## Remove Stopwords

Remove all stopwords from the text. (e.g. "I am a student" will be converted to "student")

In [24]:
# remove stopwords with nltk.corpus.stopwords
stop_words = set(stopwords.words('english'))

df_preprocessed['text'] = df_preprocessed['text'].apply(lambda x: ' '.join([word for word in word_tokenize(x) if word not in stop_words]))

In [25]:
df_preprocessed.sample(5)

,text,label
1346,saral home cotton small printed multi use runn...,Household
5893,saf framed painting wood inch x inch set paint...,Household
2153,obesity code unlocking secrets weight loss,Books
2426,ritu plastic ice cream scoop multicolour ritu ...,Household
492,scotch titanium kitchen scissor red often feel...,Household


## Tokenization

Tokenize the text into words. (e.g. "I am a student" will be converted to ["I", "am", "a", "student"])

In [26]:
# Tokenization
df_preprocessed['text'] = df_preprocessed['text'].apply(lambda x: word_tokenize(x))

In [27]:
df_preprocessed.sample(5)

,text,label
7017,"[goldstroms, women, half, slip, tank, top, cam...",Clothing & Accessories
10434,"[taparia, dep, double, ended, spanner, set, ma...",Household
7616,"[p, home, decor, carpetrunner, mat, size, x, f...",Household
8116,"[small, gods, novel, discworld, review, surely...",Household
6571,"[sencer, e, bluetooth, true, wireless, earphon...",Electronics


## Lemmatization

Lemmatize the text to convert words to their base form. (e.g. "running" will be converted to "run")

In [28]:
# Lemmatization
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

df_preprocessed['text'] = df_preprocessed['text'].apply(lambda x: [lemmatizer.lemmatize(word) for word in x])

df_preprocessed.sample(5)

,text,label
3725,"[kaveh, iran, harp, glass, butter, dish, cm, c...",Household
6275,"[deco, home, piece, set, cotton, duvet, sham, ...",Household
2218,"[decal, creation, switchboard, sticker, wall, ...",Household
3552,"[sweetea, kashmir, kahwa, basic, gm, single, s...",Books
11877,"[deckup, turrano, door, shoe, rack, dark, weng...",Household


## Split Dataset

In [29]:
X = df_preprocessed['text'].apply(lambda x: ' '.join(x))
y = df_preprocessed['label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Vectorization

## Count Vectorization

In [30]:
CV_model = CountVectorizer()

X_train_CV = CV_model.fit_transform(X_train)
X_test_CV = CV_model.transform(X_test)

## TF-IDF Vectorization

In [31]:
TFIDF_model = TfidfVectorizer()

X_train_TFIDF = TFIDF_model.fit_transform(X_train)
X_test_TFIDF = TFIDF_model.transform(X_test)

# Model Building

## Random Forest Classifier

In [32]:
rf_CV = RandomForestClassifier(criterion='entropy', n_estimators=100, random_state=42, min_samples_split=2, min_samples_leaf=2)

rf_CV.fit(X_train_CV, y_train)

y_pred_CV = rf_CV.predict(X_test_CV)

print(classification_report(y_test, y_pred_CV))

                        precision    recall  f1-score   support

                 Books       0.93      0.93      0.93       611
Clothing & Accessories       0.98      0.92      0.95       440
           Electronics       0.96      0.86      0.91       518
             Household       0.88      0.96      0.92       953

              accuracy                           0.92      2522
             macro avg       0.94      0.92      0.93      2522
          weighted avg       0.93      0.92      0.92      2522



In [33]:
rf_tfidf = RandomForestClassifier(criterion='entropy', n_estimators=100, random_state=42, min_samples_split=2, min_samples_leaf=2)

rf_tfidf.fit(X_train_TFIDF, y_train)

y_pred_TFIDF = rf_tfidf.predict(X_test_TFIDF)

print(classification_report(y_test, y_pred_TFIDF))

                        precision    recall  f1-score   support

                 Books       0.94      0.94      0.94       611
Clothing & Accessories       0.97      0.91      0.94       440
           Electronics       0.96      0.85      0.90       518
             Household       0.88      0.96      0.92       953

              accuracy                           0.92      2522
             macro avg       0.94      0.91      0.92      2522
          weighted avg       0.93      0.92      0.92      2522



## SVC

In [34]:
svc_CV = SVC(kernel='linear', C=1, random_state=42)

svc_CV.fit(X_train_CV, y_train)

y_pred_svc_CV = svc_CV.predict(X_test_CV)

print(classification_report(y_test, y_pred_svc_CV))

                        precision    recall  f1-score   support

                 Books       0.90      0.94      0.92       611
Clothing & Accessories       0.96      0.94      0.95       440
           Electronics       0.94      0.92      0.93       518
             Household       0.95      0.94      0.95       953

              accuracy                           0.94      2522
             macro avg       0.94      0.94      0.94      2522
          weighted avg       0.94      0.94      0.94      2522



In [35]:
svc_tfidf = SVC(kernel='linear', C=1, random_state=42)

svc_tfidf.fit(X_train_TFIDF, y_train)

y_pred_svc_TFIDF = svc_tfidf.predict(X_test_TFIDF)

print(classification_report(y_test, y_pred_svc_TFIDF))

                        precision    recall  f1-score   support

                 Books       0.97      0.96      0.96       611
Clothing & Accessories       0.98      0.98      0.98       440
           Electronics       0.97      0.94      0.95       518
             Household       0.95      0.98      0.97       953

              accuracy                           0.97      2522
             macro avg       0.97      0.96      0.97      2522
          weighted avg       0.97      0.97      0.97      2522



# Conclusion

Overall Performance: The SVM classifier with TF-IDF vectorizer achieved the highest accuracy and F1-scores across all categories, with an accuracy of 97%. This indicates a strong model fit and suggests that SVM with TF-IDF effectively captures the nuances in text features across classes.

Class-Specific Insights:
- For the Books category, all models performed well, achieving F1-scores around 0.93–0.96. The best performance was with the SVM + TF-IDF combination, achieving a precision and recall of 0.97 and 0.96, respectively.
- Clothing & Accessories showed strong performance across models, but SVM + TF-IDF again led with an F1-score of 0.98, reflecting high precision and recall.
- Electronics had slightly lower recall scores across all models compared to other classes, suggesting this category was more challenging to classify accurately. However, SVM with TF-IDF performed best, with an F1-score of 0.95.
- Household items consistently had high recall (indicating few false negatives), particularly with SVM + TF-IDF, achieving an F1-score of 0.97.

Model Comparison:

- Random Forest models achieved satisfactory results but were generally outperformed by SVM, especially when combined with TF-IDF vectorization.
- SVM models, particularly with TF-IDF, consistently produced higher precision, recall, and F1-scores across classes, with a particularly high weighted F1-score of 0.97.

Vectorization Impact:
The TF-IDF vectorizer outperformed the Count vectorizer across both Random Forest and SVM models, suggesting that it captures more relevant feature importance for text classification in this dataset.

Recommendation: For deployment, the SVM model with TF-IDF vectorization is recommended due to its superior accuracy and balanced performance across all classes.